In [13]:
import pandas as pd

# ===== Load file =====
file_path = "micro_climate_rh_t_et0.xlsx"
df = pd.read_excel(file_path, sheet_name="micro_climate_rh_t_et0")

# ===== Keep ET0 numeric =====
df["ET0"] = pd.to_numeric(df["ET0"], errors="coerce")
df = df.dropna(subset=["ET0"]).reset_index(drop=True)

# ===== Parse only the first timestamp safely =====
first_raw = str(df.loc[0, "timestamp_dayfirst"]).strip()

# this first row in your file is in DD/MM/YYYY format
start_ts = pd.to_datetime(first_raw, format="%d/%m/%Y %H:%M:%S", errors="raise")

# ===== Rebuild the full timestamp column as a strict 10-minute sequence =====
df["timestamp_fixed"] = pd.date_range(
    start=start_ts,
    periods=len(df),
    freq="10min"
)

# ===== Sum ET0 by day =====
daily_et0 = (
    df.set_index("timestamp_fixed")
      .resample("D")["ET0"]
      .sum()
      .reset_index()
)

daily_et0.columns = ["date", "daily_et0"]

# ===== Optional display format =====
daily_et0["date"] = daily_et0["date"].dt.strftime("%d/%m/%Y")

# ===== Print result =====
print(daily_et0)

# ===== Save =====
output_file = "daily_et0_fixed.xlsx"
daily_et0.to_excel(output_file, index=False)

print(f"\nSaved to: {output_file}")

           date  daily_et0
0    29/05/2025   5.456263
1    30/05/2025   5.387276
2    31/05/2025   5.959173
3    01/06/2025   5.900478
4    02/06/2025   5.982165
..          ...        ...
111  17/09/2025   3.499936
112  18/09/2025   3.211325
113  19/09/2025   3.571398
114  20/09/2025   3.766119
115  21/09/2025   3.824716

[116 rows x 2 columns]

Saved to: daily_et0_fixed.xlsx
